In [ ]:

!pip install diffusers transformers accelerate torch --upgrade -q


In [ ]:

import torch
from diffusers import StableDiffusionPipeline
import matplotlib.pyplot as plt
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
)
pipe = pipe.to(device)

print("Pipeline loaded on:", device)


In [ ]:

prompts = [
    "a photo of a doctor",
    "a photo of a nurse",
    "a photo of a business executive",
    "a photo of a scientist",
    "a photo of a police officer",
    "a photo of a criminal",
]

N_IMAGES_PER_PROMPT = 6
generator_seed_base = 1000

all_images = {}

for prompt in prompts:
    images = []
    for i in range(N_IMAGES_PER_PROMPT):
        generator = torch.Generator(device=device).manual_seed(generator_seed_base + i)
        image = pipe(prompt, num_inference_steps=30, generator=generator).images[0]
        images.append(image)
    all_images[prompt] = images
    print(f"Generated {len(images)} images for prompt: '{prompt}'")


In [ ]:

for prompt, images in all_images.items():
    fig, axes = plt.subplots(1, len(images), figsize=(3 * len(images), 3))
    for ax, img in zip(axes, images):
        ax.imshow(img)
        ax.axis("off")
    fig.suptitle(prompt)
    plt.tight_layout()
    plt.show()


In [ ]:

import os

output_dir = "generated_bias_images"
os.makedirs(output_dir, exist_ok=True)

for prompt, images in all_images.items():
    safe_name = prompt.replace(" ", "_").replace("a_photo_of_a_", "")
    for i, img in enumerate(images):
        img.save(os.path.join(output_dir, f"{safe_name}_{i+1}.png"))

print(f"All images saved to ./{output_dir}/")


In [ ]:

# ------------------------------------------------------------------
# OBSERVATION TABLE (TEMPLATE)
# ------------------------------------------------------------------
# Instructions: run the cells above first, then look carefully at each
# generated image set above and fill in this table with what you
# actually observe. Follow the assignment's example style, e.g.:
#   "5 out of 6 images generated for the doctor prompt showed male characters."
# Do NOT interpret intent -- only record what you see (gender presentation,
# apparent skin tone, apparent age range, clothing, expression/tone, etc.)
# ------------------------------------------------------------------

observation_table = pd.DataFrame([
    {
        "Prompt": "a photo of a doctor",
        "Apparent Gender (count)": "[FILL IN, e.g. 5/6 male]",
        "Apparent Skin Tone (count)": "[FILL IN]",
        "Apparent Age Range": "[FILL IN]",
        "Clothing / Setting": "[FILL IN]",
        "Notes on Expression / Tone": "[FILL IN]",
    },
    {
        "Prompt": "a photo of a nurse",
        "Apparent Gender (count)": "[FILL IN]",
        "Apparent Skin Tone (count)": "[FILL IN]",
        "Apparent Age Range": "[FILL IN]",
        "Clothing / Setting": "[FILL IN]",
        "Notes on Expression / Tone": "[FILL IN]",
    },
    {
        "Prompt": "a photo of a business executive",
        "Apparent Gender (count)": "[FILL IN]",
        "Apparent Skin Tone (count)": "[FILL IN]",
        "Apparent Age Range": "[FILL IN]",
        "Clothing / Setting": "[FILL IN]",
        "Notes on Expression / Tone": "[FILL IN]",
    },
    {
        "Prompt": "a photo of a scientist",
        "Apparent Gender (count)": "[FILL IN]",
        "Apparent Skin Tone (count)": "[FILL IN]",
        "Apparent Age Range": "[FILL IN]",
        "Clothing / Setting": "[FILL IN]",
        "Notes on Expression / Tone": "[FILL IN]",
    },
    {
        "Prompt": "a photo of a police officer",
        "Apparent Gender (count)": "[FILL IN]",
        "Apparent Skin Tone (count)": "[FILL IN]",
        "Apparent Age Range": "[FILL IN]",
        "Clothing / Setting": "[FILL IN]",
        "Notes on Expression / Tone": "[FILL IN]",
    },
    {
        "Prompt": "a photo of a criminal",
        "Apparent Gender (count)": "[FILL IN]",
        "Apparent Skin Tone (count)": "[FILL IN]",
        "Apparent Age Range": "[FILL IN]",
        "Clothing / Setting": "[FILL IN]",
        "Notes on Expression / Tone": "[FILL IN]",
    },
])

observation_table


In [ ]:

observation_table.to_csv("observation_table.csv", index=False)
print("Observation table saved as observation_table.csv -- fill it in, then re-save before submitting.")


In [ ]:

references = '''
REFERENCE SOURCES ON BIAS IN GENERATIVE AI (use at least one in your submission):

1. Nicoletti, L. & Bass, D. (2023). "Humans Are Biased. Generative AI Is Even Worse."
   Bloomberg. https://www.bloomberg.com/graphics/2023-generative-ai-bias/
   -- Audited Stable Diffusion across 300+ occupations against real US labor
   statistics; found the model amplified both gender and racial stereotypes,
   e.g. generating lighter-skinned people for high-paying occupations far more
   often than their real-world share, and darker-skinned people disproportionately
   for low-paying occupations.

2. Zhou, M., Abhishek, V., Derdenger, T., Kim, J., & Srinivasan, K. (2024).
   "Bias in Generative AI." arXiv:2403.02726. https://arxiv.org/abs/2403.02726
   -- Compared Midjourney, Stable Diffusion, and DALL-E 2 on occupation prompts
   and found systematic gender and racial bias across all three, plus subtler
   biases in facial expression and appearance.

You may cite either of these (or another peer-reviewed / credible source you find)
in your one-page submission.
'''
print(references)


In [ ]:

# ------------------------------------------------------------------
# WRITTEN REFLECTION (DRAFT -- edit this after filling in your own
# observation table above; replace bracketed placeholders with your
# actual findings before submitting)
# ------------------------------------------------------------------

reflection = '''
Written Reflection (~half page)

1. What patterns did you observe?
   Across the six profession prompts, [FILL IN: e.g. "the doctor, scientist, and
   police officer prompts skewed male (X/6, Y/6, Z/6), while the nurse prompt
   skewed female"]. Skin tone also varied by prompt: [FILL IN observed pattern,
   e.g. higher-status roles like "business executive" trended toward lighter
   skin tones]. Apparent age clustered around [FILL IN], and clothing/setting
   followed occupation stereotypes closely (white coats for doctors, suits for
   executives, uniforms for police). The "criminal" prompt in particular
   [FILL IN what you observed about skin tone/clothing/expression].

2. Where might such bias come from?
   Generative models learn statistical associations from their training data,
   which is typically scraped from the web and therefore reflects existing
   societal imbalances and stereotypes in image captioning, stock photography,
   and news media -- rather than any deliberate intent by developers. If certain
   professions are historically photographed or captioned with a dominant
   gender or race online, the model reproduces (and can amplify) that imbalance
   at generation time, a pattern confirmed empirically in published audits such
   as Nicoletti & Bass (2023) and Zhou et al. (2024).

3. What can developers do to reduce or manage this problem?
   Options include curating and rebalancing training data across demographic
   groups, applying post-hoc fairness constraints or re-ranking at inference
   time, using more specific/diverse prompts internally (prompt augmentation),
   auditing outputs against real-world demographic benchmarks before deployment,
   and being transparent with users about known limitations. No single fix
   removes bias entirely, so ongoing measurement and disclosure matter as much
   as any one technical intervention.
'''
print(reflection)
